# BrainScanAI — Notebook 2 : approche semi-supervisée

**Le point de départ.** Le notebook 1 a produit deux jeux de données, gardés strictement séparés :
- un jeu **fortement labellisé** : 96 images annotées par des radiologues, découpées en 67 images de train et 29 de test ;
- un jeu **faiblement labellisé** : 1 214 images étiquetées automatiquement par clustering, avec une qualité estimée à 88 %.

**La question de ce notebook.** Les 1 214 images sans annotation humaine permettent-elles de construire un meilleur modèle que les 96 images expertes seules ?

**Le protocole**, qui suit la consigne du projet :

| Modèle | Entraînement | Ce qu'il mesure |
| --- | --- | --- |
| **B0 — faible seul** | Les 1 214 labels faibles | Que vaut un modèle entraîné sans aucun label humain ? |
| **B — semi-supervisé** | B0, dont l'entraînement est **poursuivi** sur les 67 images fortes | Le préentraînement apporte-t-il quelque chose ? |
| **A — supervisé (référence)** | Les 67 images fortes, depuis zéro | La référence à battre |
| **C — semi-supervisé filtré** | Comme B, mais en ne préentraînant que sur les labels faibles les plus sûrs | Vaut-il mieux beaucoup de labels bruités ou moins de labels fiables ? |

La comparaison qui répond au projet est **A contre B**, sur les mêmes 29 images de test, qu'aucun modèle n'a jamais vues.

## 1. Environnement

Ce notebook fonctionne **sur Google Colab comme en local**. La cellule ci-dessous détecte l'environnement et règle les chemins en conséquence.

**Sur Colab**, elle monte Google Drive et décompresse le dataset dans le disque temporaire de la machine (`/content`), bien plus rapide à lire que Drive. Il faut donc avoir déposé dans un dossier `BrainScanAI` de ton Drive :
- `mri_dataset_brain_cancer_oc.zip` (les images) ;
- `inventaire_images.csv` et `labels_faibles.csv` (produits par le notebook 1).

Pense aussi à activer le GPU : **Exécution > Modifier le type d'exécution > T4 GPU**.

In [ ]:
import copy
import random
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

try:
    import google.colab  # présent uniquement sur Colab
    SUR_COLAB = True
except ImportError:
    SUR_COLAB = False

if SUR_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/BrainScanAI")
    DATA_DIR = Path("/content/mri_dataset_brain_cancer_oc")
    if not DATA_DIR.exists():   # décompression une fois par session
        with zipfile.ZipFile(DRIVE_DIR / "mri_dataset_brain_cancer_oc.zip") as archive:
            archive.extractall("/content")
    INPUT_DIR = DRIVE_DIR
    OUTPUT_DIR = DRIVE_DIR / "outputs"
else:
    DATA_DIR = Path("../mri_dataset_brain_cancer_oc")
    INPUT_DIR = Path("../outputs")
    OUTPUT_DIR = Path("../outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")

# Graine fixée partout : deux exécutions donnent les mêmes résultats
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Colab : {SUR_COLAB} | calcul sur : {device}")
print(f"Données : {DATA_DIR} ({'trouvé' if DATA_DIR.exists() else 'MANQUANT'})")

## 2. Chargement des deux jeux de données

On recharge les fichiers produits par le notebook 1. Le découpage train / test du jeu fort a été **figé à l'étape 3** : on le réutilise tel quel, sans le retirer, sinon les images de test auraient pu servir au clustering.

La cellule vérifie aussi qu'**aucune image n'appartient aux deux jeux** : c'est la garantie que le jeu faible et le jeu fort restent séparés.

In [ ]:
CLASSES = {"normal": 0, "cancer": 1}   # cancer = classe positive (celle qu'il ne faut pas rater)

inventaire = pd.read_csv(INPUT_DIR / "inventaire_images.csv")
faibles = pd.read_csv(INPUT_DIR / "labels_faibles.csv")

fort = inventaire[inventaire["subset"] == "avec_labels"]
fort_train = fort[fort["split_fort"] == "train"].reset_index(drop=True)
fort_test = fort[fort["split_fort"] == "test"].reset_index(drop=True)

# Garde-fou : le jeu faible et le jeu fort ne doivent partager aucune image
assert set(faibles["path"]).isdisjoint(set(fort["path"])), "Fuite entre jeu faible et jeu fort"

pd.DataFrame({
    "jeu faible": faibles["label_faible"].value_counts(),
    "fort (train)": fort_train["label"].value_counts(),
    "fort (test)": fort_test["label"].value_counts(),
}).fillna(0).astype(int)

## 3. Préparation des images

**Taille retenue : 128×128.** Les images d'origine font 512×512, mais un CNN entraîné depuis zéro sur si peu de données n'a pas besoin d'autant de détail : une tumeur reste parfaitement visible à cette taille, et l'entraînement est 3 fois plus rapide qu'en 224.

**L'augmentation de données**, appliquée **uniquement à l'entraînement**, sert ici deux objectifs :

| Transformation | Pourquoi |
| --- | --- |
| Retournement horizontal | Une tumeur à gauche ou à droite reste une tumeur |
| Rotation de ±15° | Rend le modèle tolérant à l'inclinaison de la tête, et surtout **moins dépendant de l'orientation** — le biais identifié à l'étape 3 |
| Variation de luminosité et de contraste | Rend le modèle moins sensible au **type de séquence IRM**, l'autre biais identifié |

Avec seulement 67 images fortes, c'est aussi la meilleure protection contre le surapprentissage : le modèle ne voit jamais deux fois exactement la même image.

En évaluation, **aucune augmentation** : on mesure les performances sur les images telles qu'elles sont.

In [ ]:
IMG_SIZE = 128

transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),      # une seule dimension de couleur : ce sont des IRM
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),                             # [0, 255] -> [0, 1], format (C, H, W)
])

transform_eval = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


class MRIDataset(Dataset):
    """Lit les images sur le disque et renvoie (image transformée, label)."""

    def __init__(self, df, colonne_label, transform):
        self.paths = [DATA_DIR / chemin for chemin in df["path"]]
        self.labels = [CLASSES[label] for label in df[colonne_label]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i])), self.labels[i]


print(f"{len(MRIDataset(fort_train, 'label', transform_eval))} images dans le train fort")

**Contrôle visuel.** La première ligne montre les images redimensionnées telles que le modèle les verra en évaluation ; la seconde, les mêmes après augmentation aléatoire. Il faut vérifier que les tumeurs restent lisibles en 128×128.

In [ ]:
echantillon = fort_train.sample(6, random_state=0)

fig, axes = plt.subplots(2, len(echantillon), figsize=(2.4 * len(echantillon), 5.2))
for col, (_, ligne) in enumerate(echantillon.iterrows()):
    image = Image.open(DATA_DIR / ligne["path"])
    axes[0, col].imshow(transform_eval(image)[0], cmap="gray")
    axes[0, col].set_title(ligne["label"], fontsize=9)
    axes[1, col].imshow(transform_train(image)[0], cmap="gray")
    for ligne_axes in axes[:, col]:
        ligne_axes.axis("off")
axes[0, 0].text(-0.12, 0.5, "128x128", transform=axes[0, 0].transAxes, rotation=90,
                va="center", ha="right", fontweight="bold")
axes[1, 0].text(-0.12, 0.5, "augmentée", transform=axes[1, 0].transAxes, rotation=90,
                va="center", ha="right", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. L'architecture du CNN

Le modèle est volontairement **petit**, parce qu'on n'a que 67 images labellisées : un gros réseau les mémoriserait au lieu d'apprendre.

```
Entrée : 1 canal, 128×128
 ├─ Bloc 1 : conv 3×3 (32 filtres) → batch norm → ReLU → max pooling 2×2   → 64×64
 ├─ Bloc 2 : conv 3×3 (64 filtres) → batch norm → ReLU → max pooling 2×2   → 32×32
 ├─ Bloc 3 : conv 3×3 (128 filtres) → batch norm → ReLU → max pooling 2×2  → 16×16
 ├─ Global average pooling                                                  → 128 valeurs
 ├─ Dropout (30 %)
 └─ Couche dense 128 → 2                                                    → cancer / normal
```

Le rôle de chaque brique :

| Brique | Rôle |
| --- | --- |
| **Convolution 3×3** | Détecte des motifs locaux. Les filtres sont partagés sur toute l'image, ce qui fait peu de paramètres et respecte la structure spatiale |
| **Batch normalization** | Recentre les valeurs à chaque couche, ce qui stabilise l'entraînement et laisse passer le gradient |
| **ReLU** | La non-linéarité. Sa pente vaut 1 du côté positif, donc le gradient traverse sans s'affaiblir |
| **Max pooling** | Divise la taille par deux, élargit la zone vue par les couches suivantes et rend le modèle tolérant aux petits décalages |
| **Global average pooling** | Remplace un aplatissement classique : il fait la moyenne de chaque carte, ce qui réduit énormément le nombre de paramètres du classifieur |
| **Dropout** | Désactive au hasard 30 % des valeurs pendant l'entraînement, pour éviter que le modèle ne s'appuie sur quelques détails |
| **Couche dense finale** | Le **classifieur** : elle combine les 128 caractéristiques en deux scores, un par classe |

Le nombre de filtres double à chaque bloc (32, 64, 128), au fur et à mesure que la taille de l'image diminue : c'est le schéma classique des CNN.

In [ ]:
class PetitCNN(nn.Module):
    """Petit CNN : 3 blocs convolutifs pour extraire, une couche dense pour décider."""

    def __init__(self, n_classes=2, p_dropout=0.3):
        super().__init__()

        def bloc(canaux_entree, canaux_sortie):
            return nn.Sequential(
                nn.Conv2d(canaux_entree, canaux_sortie, kernel_size=3, padding=1),
                nn.BatchNorm2d(canaux_sortie),
                nn.ReLU(),
                nn.MaxPool2d(2),
            )

        self.extracteur = nn.Sequential(bloc(1, 32), bloc(32, 64), bloc(64, 128))
        self.classifieur = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # moyenne de chaque carte -> 128 valeurs
            nn.Flatten(),
            nn.Dropout(p_dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifieur(self.extracteur(x))


modele_test = PetitCNN()
n_params = sum(p.numel() for p in modele_test.parameters())
print(f"{n_params:,} paramètres".replace(",", " "))
print(modele_test)

## 5. Entraînement et évaluation

**La fonction d'entraînement** suit la boucle classique du cours : passage avant, calcul de la perte, rétropropagation, mise à jour des poids.

- **Perte** : entropie croisée, la log-vraisemblance négative du chapitre 1.
- **Optimiseur** : Adam, une variante de la descente de gradient qui adapte le pas à chaque paramètre.
- **Arrêt anticipé** (*early stopping*) : plutôt que de fixer le nombre d'epochs à l'avance, on surveille le F1 sur un jeu de validation et on **garde les poids du meilleur moment**. Si le score ne progresse plus pendant plusieurs epochs, on arrête. C'est la protection la plus efficace contre le surapprentissage quand les données sont rares.

**L'évaluation** renvoie la probabilité de cancer pour chaque image, dont on tire les trois métriques du projet : accuracy, F1 sur la classe cancer, et AUC.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, roc_curve


@torch.no_grad()
def predire(model, dataset, batch_size=64):
    """Renvoie (vrais labels, probabilité prédite de cancer) pour tout le dataset."""
    model.eval()
    probas, vrais = [], []
    for x, y in DataLoader(dataset, batch_size=batch_size):
        scores = torch.softmax(model(x.to(device)), dim=1)[:, 1]   # colonne 1 = cancer
        probas.append(scores.cpu())
        vrais.append(y)
    return torch.cat(vrais).numpy(), torch.cat(probas).numpy()


def metriques(vrais, probas, seuil=0.5):
    """Les trois métriques du projet, avec le cancer comme classe positive."""
    predictions = (probas >= seuil).astype(int)
    return {
        "accuracy": accuracy_score(vrais, predictions),
        "f1": f1_score(vrais, predictions, zero_division=0),
        "auc": roc_auc_score(vrais, probas) if len(set(vrais)) > 1 else np.nan,
    }


def entrainer(model, train_ds, val_ds=None, epochs=40, batch_size=32, lr=1e-3, patience=8):
    """Entraîne le modèle. Avec un jeu de validation, applique l'arrêt anticipé sur le F1."""
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    optimiseur = torch.optim.Adam(model.parameters(), lr=lr)
    fonction_perte = nn.CrossEntropyLoss()

    meilleur = {"f1": -1.0, "poids": None, "epoch": 0}
    historique = []

    for epoch in range(1, epochs + 1):
        model.train()
        perte_totale = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimiseur.zero_grad()
            perte = fonction_perte(model(x), y)
            perte.backward()
            optimiseur.step()
            perte_totale += perte.item() * len(y)

        ligne = {"epoch": epoch, "perte_train": perte_totale / len(train_ds)}

        if val_ds is not None:
            scores = metriques(*predire(model, val_ds))
            ligne.update({f"val_{cle}": valeur for cle, valeur in scores.items()})
            if scores["f1"] > meilleur["f1"]:
                # Nouveau meilleur modèle : on met ses poids de côté
                meilleur = {"f1": scores["f1"], "poids": copy.deepcopy(model.state_dict()), "epoch": epoch}
            elif epoch - meilleur["epoch"] >= patience:
                historique.append(ligne)
                break

        historique.append(ligne)

    if meilleur["poids"] is not None:
        model.load_state_dict(meilleur["poids"])   # on restaure le meilleur moment

    return pd.DataFrame(historique)

## 6. Le protocole d'évaluation

C'est le point le plus important du notebook, parce qu'avec 96 images labellisées, un protocole bâclé produit des chiffres qui ne veulent rien dire.

**Deux niveaux :**

| Niveau | Données | À quoi ça sert |
| --- | --- | --- |
| **Validation croisée à 5 plis**, dans les 67 images du train | Le train fort, découpé en 5 parts | Régler les hyperparamètres et **comparer les modèles avec une marge d'incertitude** (moyenne et écart-type sur 5 mesures) |
| **Test final** | Les 29 images de test | Mesurer une seule fois, sur des images qu'aucune étape n'a jamais vues |

**Pourquoi la validation croisée est indispensable ici.** Avec 29 images de test, une seule erreur déplace l'accuracy de 3,5 points. Un écart de 0,04 entre deux modèles peut n'être qu'une image de différence, donc du hasard. Les 5 plis donnent un écart-type, qui dit si la différence observée est réelle.

Le découpage des plis est **stratifié** : chaque pli garde la même proportion de cancers et de normaux.

In [ ]:
from sklearn.model_selection import StratifiedKFold

N_PLIS = 5


def validation_croisee(poids_initiaux=None, df=None, n_plis=N_PLIS, **hyperparams):
    """Entraîne et évalue le modèle sur n plis du train fort.

    poids_initiaux : état d'un modèle préentraîné sur les labels faibles (None = depuis zéro).
    """
    df = fort_train if df is None else df
    decoupage = StratifiedKFold(n_splits=n_plis, shuffle=True, random_state=SEED)
    resultats = []

    for pli, (idx_train, idx_val) in enumerate(decoupage.split(df, df["label"]), start=1):
        torch.manual_seed(SEED)              # même initialisation d'un modèle à l'autre
        model = PetitCNN().to(device)
        if poids_initiaux is not None:
            model.load_state_dict(poids_initiaux)   # on POURSUIT l'entraînement du modèle préentraîné

        train_ds = MRIDataset(df.iloc[idx_train], "label", transform_train)
        val_ds = MRIDataset(df.iloc[idx_val], "label", transform_eval)
        entrainer(model, train_ds, val_ds, **hyperparams)

        resultats.append({"pli": pli, **metriques(*predire(model, val_ds))})

    return pd.DataFrame(resultats)

## 7. Réglage des hyperparamètres

On teste une petite grille sur le **modèle supervisé**, par validation croisée. Chaque combinaison est évaluée sur les 5 plis, et on retient celle qui obtient le meilleur F1 moyen.

| Hyperparamètre | Valeurs testées | Ce que ça change |
| --- | --- | --- |
| **Taille de batch** | 16, 32 | Le nombre d'images vues avant chaque mise à jour des poids. Un petit batch donne un gradient plus bruité, ce qui aide parfois à généraliser ; un gros batch est plus stable et plus rapide |
| **Learning rate** | 0,001 et 0,0003 | La taille du pas de la descente de gradient. Trop grand, on enjambe le minimum ; trop petit, l'apprentissage traîne |
| **Nombre d'epochs** | Jusqu'à 40, avec arrêt anticipé | Déterminé automatiquement par l'arrêt anticipé, plutôt que fixé à l'avance |

La grille reste volontairement petite : avec 67 images, tester cinquante combinaisons reviendrait à choisir des réglages sur du bruit.

In [ ]:
grille = [{"batch_size": b, "lr": lr} for b in [16, 32] for lr in [1e-3, 3e-4]]

lignes = []
for hyperparams in grille:
    scores = validation_croisee(**hyperparams, epochs=40)
    lignes.append({
        **hyperparams,
        "f1 moyen": scores["f1"].mean(),
        "f1 écart-type": scores["f1"].std(),
        "accuracy moyenne": scores["accuracy"].mean(),
        "auc moyenne": scores["auc"].mean(),
    })

resultats_grille = pd.DataFrame(lignes).sort_values("f1 moyen", ascending=False).reset_index(drop=True)
MEILLEURS_HP = {"batch_size": int(resultats_grille.loc[0, "batch_size"]),
                "lr": float(resultats_grille.loc[0, "lr"]),
                "epochs": 40}
print("Hyperparamètres retenus :", MEILLEURS_HP)
resultats_grille.round(3)

## 8. Préentraînement sur les labels faibles

C'est le premier temps de la consigne : entraîner un CNN sur le jeu faiblement labellisé.

Deux modèles sont préentraînés :
- **B0** sur les **1 214 labels faibles** ;
- **C0** sur les **70 % les plus sûrs** seulement.

**Comment on mesure la confiance d'un label faible.** Chaque image possède un `score_cancer`, calculé à l'étape 3 : la différence entre sa distance au centre du cluster « normal » et sa distance au centre du cluster « cancer ». Un score proche de zéro signifie que l'image est à la frontière entre les deux clusters, donc que son étiquette est douteuse. On garde les images dont la valeur absolue du score est la plus élevée.

C'est la logique du **seuil de confiance** de la pseudo-labellisation : ne réutiliser que les prédictions dont le modèle est sûr, pour éviter la propagation des erreurs.

Ici pas de validation croisée : ce préentraînement ne touche pas au jeu fort, et son seul rôle est de fournir un point de départ aux modèles B et C.

In [ ]:
PART_CONSERVEE = 0.70   # part des labels faibles gardés pour le modèle C

# Les labels faibles les plus sûrs : ceux dont le score est le plus éloigné de la frontière
limite = faibles["score_cancer"].abs().quantile(1 - PART_CONSERVEE)
faibles_surs = faibles[faibles["score_cancer"].abs() >= limite].reset_index(drop=True)

print(f"Jeu faible complet : {len(faibles)} images")
print(f"Jeu faible filtré  : {len(faibles_surs)} images (seuil sur |score| = {limite:.2f})")
faibles_surs["label_faible"].value_counts()

In [ ]:
def pretrainer(df_faibles, epochs=15):
    """Entraîne un CNN depuis zéro sur des labels faibles et renvoie ses poids."""
    torch.manual_seed(SEED)
    model = PetitCNN().to(device)
    dataset = MRIDataset(df_faibles, "label_faible", transform_train)
    historique = entrainer(model, dataset, val_ds=None, epochs=epochs,
                           batch_size=MEILLEURS_HP["batch_size"], lr=MEILLEURS_HP["lr"])
    return model, historique


modele_B0, hist_B0 = pretrainer(faibles)
modele_C0, hist_C0 = pretrainer(faibles_surs)

poids_B0 = copy.deepcopy(modele_B0.state_dict())
poids_C0 = copy.deepcopy(modele_C0.state_dict())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist_B0["epoch"], hist_B0["perte_train"], marker="o", label=f"B0 — {len(faibles)} labels faibles")
ax.plot(hist_C0["epoch"], hist_C0["perte_train"], marker="o", label=f"C0 — {len(faibles_surs)} labels filtrés")
ax.set_xlabel("Epoch")
ax.set_ylabel("Perte d'entraînement")
ax.set_title("Préentraînement sur les labels faibles")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

**Évaluation de B0 sur le jeu de test.** C'est le deuxième temps de la consigne : que vaut un modèle qui n'a jamais vu le moindre label humain ?

Attention à la lecture : ce modèle a appris à reproduire les clusters, et les clusters sont justes à environ 88 % seulement. Son plafond de performance est donc limité par la qualité de ses étiquettes.

In [ ]:
test_ds = MRIDataset(fort_test, "label", transform_eval)

scores_B0 = metriques(*predire(modele_B0, test_ds))
scores_C0 = metriques(*predire(modele_C0, test_ds))

pd.DataFrame({"B0 (tous les labels faibles)": scores_B0,
              "C0 (labels faibles filtrés)": scores_C0}).round(3)

## 9. Les trois modèles en validation croisée

On compare maintenant les trois approches sur les **mêmes 5 plis** du train fort, avec les mêmes hyperparamètres. Seul le **point de départ** change :

- **A** part de poids aléatoires ;
- **B** part des poids de B0 ;
- **C** part des poids de C0.

C'est bien « le même modèle dont on poursuit l'entraînement », comme le demande la consigne.

In [ ]:
comparaison = {}
for nom, poids in [("A — supervisé", None),
                   ("B — semi-supervisé", poids_B0),
                   ("C — semi-supervisé filtré", poids_C0)]:
    comparaison[nom] = validation_croisee(poids_initiaux=poids, **MEILLEURS_HP)
    print(f"{nom} : terminé")

resume_cv = pd.DataFrame({
    nom: {
        "F1 moyen": scores["f1"].mean(),
        "F1 écart-type": scores["f1"].std(),
        "Accuracy moyenne": scores["accuracy"].mean(),
        "AUC moyenne": scores["auc"].mean(),
    }
    for nom, scores in comparaison.items()
}).T
resume_cv.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.boxplot([scores["f1"] for scores in comparaison.values()],
           tick_labels=[nom.split(" — ")[0] for nom in comparaison])
for i, scores in enumerate(comparaison.values(), start=1):
    ax.scatter([i] * len(scores), scores["f1"], alpha=0.6)   # les 5 plis, un point chacun
ax.set_ylabel("F1 (classe cancer)")
ax.set_title(f"F1 sur les {N_PLIS} plis de validation croisée")
ax.grid(alpha=0.3, axis="y")
plt.show()

## 10. Entraînement final et évaluation sur le jeu de test

Les modèles sont réentraînés sur **l'intégralité des 67 images** du train fort, puis évalués **une seule fois** sur les 29 images de test.

Pour l'arrêt anticipé, on met 20 % du train de côté comme jeu de validation interne : le test reste ainsi totalement inutilisé jusqu'à la mesure finale.

In [ ]:
from sklearn.model_selection import train_test_split

# Jeu de validation interne, uniquement pour l'arrêt anticipé
sous_train, sous_val = train_test_split(fort_train, test_size=0.2,
                                        stratify=fort_train["label"], random_state=SEED)

modeles_finaux, resultats_test = {}, {}
for nom, poids in [("A — supervisé", None),
                   ("B — semi-supervisé", poids_B0),
                   ("C — semi-supervisé filtré", poids_C0)]:
    torch.manual_seed(SEED)
    model = PetitCNN().to(device)
    if poids is not None:
        model.load_state_dict(poids)
    entrainer(model,
              MRIDataset(sous_train, "label", transform_train),
              MRIDataset(sous_val, "label", transform_eval),
              **MEILLEURS_HP)
    modeles_finaux[nom] = model
    vrais, probas = predire(model, test_ds)
    resultats_test[nom] = {**metriques(vrais, probas), "probas": probas, "vrais": vrais}

tableau_test = pd.DataFrame({
    nom: {cle: valeur for cle, valeur in scores.items() if cle in ["accuracy", "f1", "auc"]}
    for nom, scores in resultats_test.items()
}).T
tableau_test.loc["B0 — faible seul"] = scores_B0
tableau_test.round(3)

## 11. Analyse des performances

Trois lectures complémentaires :
- la **matrice de confusion**, qui sépare les types d'erreurs. En médecine, les deux colonnes n'ont pas le même poids : un **faux négatif** est une tumeur manquée, un **faux positif** une fausse alerte qu'un radiologue lèvera ;
- la **courbe ROC**, qui montre le compromis entre détection et fausses alertes pour tous les seuils possibles. L'AUC en est l'aire ;
- l'**affichage des erreurs**, pour comprendre sur quoi le modèle se trompe.

In [ ]:
fig, axes = plt.subplots(1, len(resultats_test), figsize=(4.5 * len(resultats_test), 4))
for ax, (nom, scores) in zip(axes, resultats_test.items()):
    matrice = confusion_matrix(scores["vrais"], (scores["probas"] >= 0.5).astype(int))
    ax.imshow(matrice, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, matrice[i, j], ha="center", va="center", fontsize=16)
    ax.set_xticks([0, 1], ["prédit normal", "prédit cancer"])
    ax.set_yticks([0, 1], ["vrai normal", "vrai cancer"])
    ax.set_title(f"{nom}\nF1 = {scores['f1']:.2f}", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
for nom, scores in resultats_test.items():
    fpr, tpr, _ = roc_curve(scores["vrais"], scores["probas"])
    ax.plot(fpr, tpr, marker=".", label=f"{nom} (AUC = {scores['auc']:.2f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="hasard")
ax.set_xlabel("Taux de faux positifs")
ax.set_ylabel("Taux de vrais positifs (rappel)")
ax.set_title("Courbes ROC sur le jeu de test")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Erreurs du meilleur modèle : quelles images sont mal classées, et dans quel sens ?
meilleur_nom = max(resultats_test, key=lambda nom: resultats_test[nom]["f1"])
scores = resultats_test[meilleur_nom]
predictions = (scores["probas"] >= 0.5).astype(int)

erreurs = fort_test.copy()
erreurs["proba_cancer"] = scores["probas"]
erreurs["prediction"] = np.where(predictions == 1, "cancer", "normal")
erreurs = erreurs[erreurs["prediction"] != erreurs["label"]]

print(f"Meilleur modèle : {meilleur_nom} — {len(erreurs)} erreurs sur {len(fort_test)} images de test")

if len(erreurs):
    fig, axes = plt.subplots(1, len(erreurs), figsize=(3 * len(erreurs), 3.6), squeeze=False)
    for ax, (_, ligne) in zip(axes[0], erreurs.iterrows()):
        ax.imshow(Image.open(DATA_DIR / ligne["path"]), cmap="gray")
        type_erreur = "faux négatif" if ligne["label"] == "cancer" else "faux positif"
        ax.set_title(f"{type_erreur}\nvrai : {ligne['label']} | p(cancer) = {ligne['proba_cancer']:.2f}",
                     fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

erreurs[["path", "label", "prediction", "proba_cancer"]]

## 12. Sauvegarde des résultats

In [ ]:
tableau_test.to_csv(OUTPUT_DIR / "resultats_test.csv")
resume_cv.to_csv(OUTPUT_DIR / "resultats_validation_croisee.csv")
resultats_grille.to_csv(OUTPUT_DIR / "resultats_hyperparametres.csv", index=False)
torch.save({nom: model.state_dict() for nom, model in modeles_finaux.items()},
           OUTPUT_DIR / "modeles_finaux.pt")

print("Résultats sauvegardés dans", OUTPUT_DIR)

## 13. Observations et definition of done

*À rédiger après exécution.*